In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
from config import CFG

In [ ]:
CFG.PROJECT_ROOT

In [ ]:
from utils import parse_llm_response

In [ ]:
import mlflow
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage

In [ ]:
mlflow.langchain.autolog()

In [ ]:
model = init_chat_model(
    model = CFG.BASE_MODEL,
    model_provider=CFG.MODEL_PROVIDER,
    base_url=CFG.VLLM_BASE_URL,
    api_key=CFG.VLLM_API_KEY,
    temperature=CFG.GENERATION_TEMPERATURE
)

In [ ]:
model.model_name

In [ ]:
experiment = mlflow.set_experiment("chat-app")

## Chat 

In [ ]:
system_msg = SystemMessage(
    f"""
    You are a Data Scientist with deep expertise in elections data and an SQL Expert.
    Based on a database with tables {CFG.ALLOWED_TABLES} among others, you directly answer questions about elections only.
    
    First, you classify the user's election query into one category:
        - AGGREGATION: Sums, counts, averages (e.g., 'total votes', 'turnout average').
        - RANKING: Comparisons, top/bottom lists (e.g., 'who won', 'best party').
        - CHART: Requests for distribution, trends, or visualizations.
        - GENERAL: Simple fact retrieval.
        - INVALID: Not about the election.
    
    Expected Output: Generate a single valid SQL SELECT statement. 
        - Always include a LIMIT {CFG.SQL_MAX_LIMIT}.
    """
)

human_msg = HumanMessage("Who won in Commune Cocody?")

messages = [system_msg, human_msg]

In [ ]:
%%time

response = model.invoke(messages)

raw_out = response.content
thinking, sql_query = parse_llm_response(raw_out)

print(thinking)
print(sql_query)

## Follow-up 

In [ ]:
messages.append(response)
messages.append(HumanMessage(
    "How many votes did candidate Ahmed Bamba get and what was the total number of votes?"
))

In [ ]:
%%time

response = model.invoke(messages)
print(response.content)